# Assignment 2: Embeddings and Hidden Markov Models

**Dataset:** CoNLL-2000 (`eriktks/conll2000`) — only the `tokens` and `pos_tags` fields are used.

**Notebook outline**
0. Setup, data loading, train/validation split, preprocessing & vocabulary
1. Sparse and dense word representations (PMI, TF-IDF, Word2Vec)
2. HMM POS tagger with Viterbi decoding
3. Embedding-assisted HMM

## 0. Setup

All randomness goes through one seeded generator so every run produces the same split.

In [ ]:
import re
from collections import Counter

import numpy as np
from datasets import load_dataset

SEED = 42
VALIDATION_FRACTION = 0.10   # share of the official train partition held out for tuning
MIN_COUNT = 2                # words seen fewer times in training are mapped to <UNK>

UNK_TOKEN = "<UNK>"
NUM_TOKEN = "<NUM>"

rng = np.random.default_rng(SEED)

## 0.1 Load the raw data

The dataset repository ships a loading script, which recent versions of `datasets` no longer execute,
so we load Hugging Face's auto-converted Parquet files (`refs/convert/parquet`) — the same data.

POS tags are stored as integer class ids; we decode them to tag strings and discard `chunk_tags`.

In [ ]:
def decode_tags(tag_ids, tag_names):
    """Convert a list of integer tag ids into tag strings (e.g. 19 -> 'NN')."""
    tags = []
    for tag_id in tag_ids:
        tags.append(tag_names[tag_id])
    return tags


def load_conll2000():
    """Load CoNLL-2000 and return {'train': sentences, 'test': sentences}.

    Each sentence is a pair (tokens, tags). Chunk tags are not used.
    """
    raw = load_dataset("eriktks/conll2000", revision="refs/convert/parquet")
    tag_names = raw["train"].features["pos_tags"].feature.names

    data = {}
    for split_name in ["train", "test"]:
        sentences = []
        for row in raw[split_name]:
            tokens = row["tokens"]
            tags = decode_tags(row["pos_tags"], tag_names)
            sentences.append((tokens, tags))
        data[split_name] = sentences
    return data


data = load_conll2000()
print("train sentences:", len(data["train"]))
print("test sentences: ", len(data["test"]))

first_tokens, first_tags = data["train"][0]
print("\nFirst training sentence (first 8 words):")
for i in range(8):
    print(f"  {first_tokens[i]:<12} {first_tags[i]}")

## 0.2 Train / validation split

The official **test** partition is kept untouched for final evaluation only.
We shuffle the official **train** partition with a fixed seed and hold out 10% as a validation set,
which is used for all hyperparameter tuning.

In [ ]:
def train_validation_split(sentences, validation_fraction, rng):
    """Shuffle the sentences with a seeded generator and split them into (train, validation)."""
    shuffled_indices = rng.permutation(len(sentences))
    n_validation = round(len(sentences) * validation_fraction)

    validation_sentences = []
    train_sentences = []
    for position, index in enumerate(shuffled_indices):
        if position < n_validation:
            validation_sentences.append(sentences[index])
        else:
            train_sentences.append(sentences[index])
    return train_sentences, validation_sentences


def count_tokens(sentences):
    """Total number of tokens in a list of (tokens, tags) pairs."""
    total = 0
    for tokens, tags in sentences:
        total += len(tokens)
    return total


train_sents, val_sents = train_validation_split(data["train"], VALIDATION_FRACTION, rng)
test_sents = data["test"]

splits = {
    "train": train_sents,
    "validation": val_sents,
    "test": test_sents,
}

for split_name in splits:
    sentences = splits[split_name]
    print(f"{split_name:<11} {len(sentences):>5} sentences  {count_tokens(sentences):>6} tokens")

## 0.3 Preprocessing

We keep **two views** of the same sentences, because the two halves of the assignment need different things:

| View | Used by | Processing | Why |
|---|---|---|---|
| **HMM view** | Parts 2 & 3 (tagging) | Original tokens and tags, unchanged | Capitalisation (`NNP`), digits (`CD`) and punctuation (`,` `.` `:`) are strong POS cues and all carry gold tags. |
| **Embedding view** | Part 1 (PMI, TF-IDF, Word2Vec) | Lowercase, numbers → `<NUM>`, punctuation-only tokens removed | Merges `The`/`the` and thousands of distinct numbers into shared types, shrinking the vocabulary and reducing sparsity; punctuation co-occurs with everything and would only add noise to context windows. |

Rules for the embedding view:
* **Lowercasing** — sentence-initial capitalisation otherwise splits one word into two types.
* **Number normalisation** — any token made only of digits and numeric punctuation (`3.5`, `1,000`, `1\/2`, `20%`) becomes `<NUM>`.
  Individual numbers are rare and semantically interchangeable.
* **Punctuation removal** — tokens with no letter or digit (`,` `.` `--` `` ` `` `$`) are dropped.
  Clitics like `'s` and `n't` contain letters and are kept, as they are meaningful words.

In [ ]:
# A number: only digits and numeric punctuation, with at least one digit (3.5, 1,000, 1\/2, 20%).
NUMBER_PATTERN = re.compile(r"[\d.,:/\\%+-]*\d[\d.,:/\\%+-]*")
# Used to detect punctuation-only tokens (tokens with no letter or digit).
ALPHANUMERIC_PATTERN = re.compile(r"[A-Za-z0-9]")


def normalize_token(token):
    """Return the embedding-view form of a token, or None if the token should be dropped."""
    if NUMBER_PATTERN.fullmatch(token):
        return NUM_TOKEN

    is_punctuation = ALPHANUMERIC_PATTERN.search(token) is None
    if is_punctuation:
        return None

    return token.lower()


def to_embedding_view(sentences):
    """Convert (tokens, tags) pairs into lists of normalized tokens. Tags are not needed here."""
    corpus = []
    for tokens, tags in sentences:
        normalized_sentence = []
        for token in tokens:
            normalized = normalize_token(token)
            if normalized is not None:
                normalized_sentence.append(normalized)
        corpus.append(normalized_sentence)
    return corpus


embedding_corpus = {}
for split_name in splits:
    embedding_corpus[split_name] = to_embedding_view(splits[split_name])

example_tokens, example_tags = train_sents[0]
print("raw       :", " ".join(example_tokens))
print("embedding :", " ".join(embedding_corpus["train"][0]))

## 0.4 Vocabulary

The vocabulary is built **from the training split only**, so validation/test words are genuinely unseen when they are not in it.
Words seen fewer than `MIN_COUNT` times are mapped to `<UNK>`: a word seen once gives a single, very noisy context
and inflates every matrix in Part 1. `MIN_COUNT` is a hyperparameter and is revisited during validation tuning.

Ids are assigned by descending frequency (ties broken alphabetically), so the order is deterministic.

In [ ]:
def count_words(corpus):
    """Count how many times each word appears in a list of tokenized sentences."""
    word_counts = Counter()
    for sentence in corpus:
        for word in sentence:
            word_counts[word] += 1
    return word_counts


class Vocabulary:
    """Maps words to integer ids and back. Id 0 is reserved for <UNK>."""

    def __init__(self, corpus, min_count):
        word_counts = count_words(corpus)

        # Keep only words that appear at least min_count times.
        kept_words = []
        for word in word_counts:
            if word_counts[word] >= min_count:
                kept_words.append(word)

        # Order by frequency (most frequent first); ties are broken alphabetically.
        # Python's sort is stable, so sorting alphabetically first and then by count keeps that order.
        kept_words.sort()
        kept_words.sort(key=word_counts.get, reverse=True)

        self.id_to_word = [UNK_TOKEN] + kept_words
        self.word_to_id = {}
        for word_id, word in enumerate(self.id_to_word):
            self.word_to_id[word] = word_id
        self.unk_id = self.word_to_id[UNK_TOKEN]

        # Frequency of every vocabulary entry; rare words add to the <UNK> count.
        self.counts = np.zeros(len(self.id_to_word), dtype=np.int64)
        for word in word_counts:
            self.counts[self.get_id(word)] += word_counts[word]

    def __len__(self):
        return len(self.id_to_word)

    def __contains__(self, word):
        return word in self.word_to_id

    def get_id(self, word):
        """Id of a word, or the <UNK> id if the word is not in the vocabulary."""
        if word in self.word_to_id:
            return self.word_to_id[word]
        return self.unk_id

    def encode(self, sentence):
        """Convert a list of words into a list of ids."""
        ids = []
        for word in sentence:
            ids.append(self.get_id(word))
        return ids

    def decode(self, ids):
        """Convert a list of ids back into words."""
        words = []
        for word_id in ids:
            words.append(self.id_to_word[word_id])
        return words


vocab = Vocabulary(embedding_corpus["train"], MIN_COUNT)

encoded_corpus = {}
for split_name in embedding_corpus:
    encoded_sentences = []
    for sentence in embedding_corpus[split_name]:
        encoded_sentences.append(vocab.encode(sentence))
    encoded_corpus[split_name] = encoded_sentences

print(f"vocabulary size (including {UNK_TOKEN}): {len(vocab)}")
print("15 most frequent words:", vocab.id_to_word[1:16])

### Vocabulary coverage

How many running tokens fall outside the vocabulary in each split (embedding view).

In [ ]:
def unk_rate(encoded_sentences, unk_id):
    """Fraction of tokens that were mapped to <UNK>."""
    n_unknown = 0
    n_total = 0
    for sentence in encoded_sentences:
        for word_id in sentence:
            n_total += 1
            if word_id == unk_id:
                n_unknown += 1
    return n_unknown / n_total


n_train_types = len(count_words(embedding_corpus["train"]))
print(f"distinct training words: {n_train_types}")
print(f"kept in vocabulary (count >= {MIN_COUNT}): {len(vocab) - 1}")
print()

for split_name in encoded_corpus:
    rate = unk_rate(encoded_corpus[split_name], vocab.unk_id)
    print(f"{split_name:<11} <UNK> rate: {rate:6.2%}")

### Unknown words for the HMM

For tagging, a word is *unknown* when its exact surface form never appears in the training split.
We record this now so that Parts 2 and 3 can report unknown-word results separately.

In [ ]:
def collect_words_and_tags(sentences):
    """Return the set of distinct words and the set of distinct tags in (tokens, tags) pairs."""
    words = set()
    tags = set()
    for sentence_tokens, sentence_tags in sentences:
        for token in sentence_tokens:
            words.add(token)
        for tag in sentence_tags:
            tags.add(tag)
    return words, tags


def count_unknown_tokens(sentences, known_words):
    """Return (number of tokens not in known_words, total number of tokens)."""
    n_unknown = 0
    n_total = 0
    for tokens, tags in sentences:
        for token in tokens:
            n_total += 1
            if token not in known_words:
                n_unknown += 1
    return n_unknown, n_total


hmm_train_words, train_tag_set = collect_words_and_tags(train_sents)
all_tags = sorted(train_tag_set)

print(f"HMM training word types: {len(hmm_train_words)}")
print(f"POS tags: {len(all_tags)}")
print()

for split_name in ["validation", "test"]:
    n_unknown, n_total = count_unknown_tokens(splits[split_name], hmm_train_words)
    print(f"{split_name:<11} unknown tokens: {n_unknown:>5} / {n_total} ({n_unknown / n_total:.2%})")

### Summary of objects produced in this section

| Name | Contents |
|---|---|
| `train_sents`, `val_sents`, `test_sents` | HMM view: lists of `(tokens, tags)` |
| `embedding_corpus[split]` | Embedding view: lists of normalised tokens |
| `vocab` | `Vocabulary` built from the training embedding view |
| `encoded_corpus[split]` | Embedding view as lists of vocabulary ids |
| `hmm_train_words` | Surface forms seen in training (defines HMM unknown words) |